# Compass Dashboard

In [2]:
import pandas as pd

In [45]:
shift_data = {
    'month':['march','march','march'],
    'day':['Sunday','Monday','Tuesday'],
    'start':[1445,1445,1445],
    'end':[1835,1835,1835]
}

shifts = pd.DataFrame(shift_data)
shifts

,month,day,start,end
0,march,Sunday,1445,1835
1,march,Monday,1445,1835
2,march,Tuesday,1445,1835


In [46]:
fare_data = {
    '1zone': 2.70,
    '2zone': 4.00,
    '1zonemonth': 111.60,
    '2zonemonth': 149.25,
    '2zoneaddfare': 1.50,
    'yvraddfare' : 5.00
}
fare_data

{'1zone': 2.7,
 '2zone': 4.0,
 '1zonemonth': 111.6,
 '2zonemonth': 149.25,
 '2zoneaddfare': 1.5,
 'yvraddfare': 5.0}

In [48]:
def ShiftsToTrips(df:pd.DataFrame):
    "Convert a shifts schedule to the implied commute schedule"
    trips = df.copy()
    trips['start'] = df['start'] - 100
    trips.rename(columns={'start':'home','end':'YVR'}, inplace=True)
    trips = trips.melt(id_vars=['month','day'],
                        value_vars=['home','YVR'],
                        var_name='origin',
                        value_name='time')
    return trips

In [49]:
ShiftsToTrips(shifts)

,month,day,origin,time
0,march,Sunday,home,1345
1,march,Monday,home,1345
2,march,Tuesday,home,1345
3,march,Sunday,YVR,1835
4,march,Monday,YVR,1835
5,march,Tuesday,YVR,1835


In [ ]:
def GetOneZoneTrips(df:pd.DataFrame):
    "Find all trips that occur on a weekend or after 6:30pm"
    one_zone = df.loc[(df['day'].isin(['Saturday','Sunday'])) | (df['time']>1800)]

    return one_zone

GetOneZoneTrips(trips)

,month,day,origin,time
0,march,Sunday,home,1345
3,march,Sunday,YVR,1835
4,march,Monday,YVR,1835
5,march,Tuesday,YVR,1835


In [66]:
def GetTwoZoneTrips(df:pd.DataFrame):
    "find all trips that occur on a weekday before 6:30pm"
    two_zone = df.loc[(df['day'].isin(['Monday','Tuesday','Wednesday','Thursday','Friday']) & (df['time']<1800))]

    return two_zone

GetTwoZoneTrips(trips)

,month,day,origin,time
1,march,Monday,home,1345
2,march,Tuesday,home,1345


In [57]:
def GetAddFareTrips(df:pd.DataFrame):
    "find all trips that will incur the YVR add fare"
    add_fare = df.loc[df['origin'] == 'YVR']

    return add_fare

GetAddFareTrips(trips)

,month,day,origin,time
3,march,Sunday,YVR,1835
4,march,Monday,YVR,1835
5,march,Tuesday,YVR,1835


In [77]:
def TotalCost(shifts:pd.DataFrame, fares:dict, method:str):
    "find the total cost of commuting based on the given method"

    # convert shifts to trips
    trips = ShiftsToTrips(shifts)

    # count one zone trips
    OneZone = GetOneZoneTrips(trips).shape[0]

    # count two zone trips
    TwoZone = GetTwoZoneTrips(trips).shape[0]

    # count add fare trips
    AddFare = GetAddFareTrips(trips).shape[0]

    if method == 'StoredValue':
        TotalCost = OneZone*fares['1zone'] + TwoZone*fares['2zone'] + AddFare*fares['yvraddfare']
        return TotalCost
    if method == 'OneZoneMonthly':
        TotalCost = fares['1zonemonth'] + TwoZone*fares['2zoneaddfare']
        return TotalCost
    if method == 'TwoZoneMonthly':
        TotalCost = fares['2zonemonth']
        return TotalCost

    return None

TotalCost(shifts, fare_data, 'StoredValue')

33.8

In [78]:
CommuteCost = {
    'Stored Value' : TotalCost(shifts, fare_data, 'StoredValue'),
    '1 Zone Month Pass' : TotalCost(shifts, fare_data, 'OneZoneMonthly'),
    '2 Zone Month Pass' : TotalCost(shifts, fare_data, 'TwoZoneMonthly')
}

pd.DataFrame(CommuteCost, index=range(1))

,Stored Value,1 Zone Month Pass,2 Zone Month Pass
0,33.8,114.6,149.25
